# Tarea - 4 - Clasificación Binaria

Utilizando el dataset de fraude, el dueño del banco nos plantea la problemática de identificar a través de un modelo si la transacción es  hecha por una persona "adulta mayor" ( considerar a partir de 50 años ). 

Requisitos de la tarea:
* Subir el archivo .paquet solamente con la TAD. 
* Subir el archivo notebook .ipynb
* La columna con la variable objetivo se debe de llamar "y"
* Las variables finales y mejores para la matriz objetivo deben de tener el prefijo "x_"
* Construir las siguiente variables:
Proporción entre la cantidad de la transacción y la hora del día. ( colocar 0 en caso de ser la primera hora del día).
Proporción entre la cantidad de la transacción y la distancia euclidiana.
Nota importante: No colocar la columna edad ni la columna is_fraud dentro del análisis.


### LIBRERIAS

In [1]:
import pandas as pd
import numpy as np
import os
import gc ##Limpia mi memoria RAM

##ML
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score,accuracy_score,confusion_matrix
from sklearn.linear_model import LogisticRegression
import seaborn as sns


In [2]:
pd.set_option( 'display.max_columns', 200)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

### LECTURA DE DATOS

# Diccionario de datos
* index - Unique Identifier for each row
* transdatetrans_time - Transaction DateTime
* cc_num - Credit Card Number of Customer
* merchant - Merchant Name
* category - Category of Merchant
* amt - Amount of Transaction
* first - First Name of Credit Card Holder
* last - Last Name of Credit Card Holder
* gender - Gender of Credit Card Holder
* street - Street Address of Credit Card Holder
* city - City of Credit Card Holder
* state - State of Credit Card Holder
* zip - Zip of Credit Card Holder
* lat - Latitude Location of Credit Card Holder
* long - Longitude Location of Credit Card Holder
* city_pop - Credit Card Holder's City Population
* job - Job of Credit Card Holder
* dob - Date of Birth of Credit Card Holder
* trans_num - Transaction Number
* unix_time - UNIX Time of transaction
* merch_lat - Latitude Location of Merchant
* merch_long - Longitude Location of Merchant
* is_fraud - Fraud Flag <--- Target Class

#### PARTE 1: RECUPERACION DE LOS DATOS


In [23]:
!ls -l


total 489856
-rw-r--r-- 1 alex alex 150354339 Aug  5  2020 fraudTest.csv
-rw-r--r-- 1 alex alex       109 Sep 20 19:46 fraudTest.csv:Zone.Identifier
-rw-r--r-- 1 alex alex 351238196 Aug  5  2020 fraudTrain.csv
-rw-r--r-- 1 alex alex       109 Sep 20 19:46 fraudTrain.csv:Zone.Identifier


In [22]:
%cd archive/


/home/alex/DiplomadoCD/Modulo1/fraud/archive


In [ ]:
df = pd.read_csv( "fraudTrain.csv" )

In [25]:
df.head(2)

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.9700,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,NC,28654,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.0113,-82.0483,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.2300,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,WA,99160,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.1590,-118.1865,0


In [ ]:
#Como borrar memoria
##del cov_df
##import gc
## gc.collect()

192

In [26]:
df.shape

(1296675, 23)

LIMPIEZA

In [27]:
df.dtypes

Unnamed: 0                 int64
trans_date_trans_time        str
cc_num                     int64
merchant                     str
category                     str
amt                      float64
first                        str
last                         str
gender                       str
street                       str
city                         str
state                        str
zip                        int64
lat                      float64
long                     float64
city_pop                   int64
job                          str
dob                          str
trans_num                    str
unix_time                  int64
merch_lat                float64
merch_long               float64
is_fraud                   int64
dtype: object

In [28]:
for c in df.columns:
    print(c, df[c].map(type).unique().tolist() )

Unnamed: 0 [<class 'int'>]
trans_date_trans_time [<class 'str'>]
cc_num [<class 'int'>]
merchant [<class 'str'>]
category [<class 'str'>]
amt [<class 'float'>]
first [<class 'str'>]
last [<class 'str'>]
gender [<class 'str'>]
street [<class 'str'>]
city [<class 'str'>]
state [<class 'str'>]
zip [<class 'int'>]
lat [<class 'float'>]
long [<class 'float'>]
city_pop [<class 'int'>]
job [<class 'str'>]
dob [<class 'str'>]
trans_num [<class 'str'>]
unix_time [<class 'int'>]
merch_lat [<class 'float'>]
merch_long [<class 'float'>]
is_fraud [<class 'int'>]


In [29]:
### Celda de limpieza
df.drop( columns = ['Unnamed: 0'] ,inplace = True)
df.drop(columns=['unix_time'],inplace=True)

df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'], format='%Y-%m-%d %H:%M:%S')
df['dob'] = pd.to_datetime(df['dob'], format='%Y-%m-%d')
df['zip'] = df['zip'].apply(lambda x: str(x).zfill(5) )

In [30]:
df[['category','is_fraud']].value_counts().reset_index()

,category,is_fraud,count
0,gas_transport,0,131041
1,home,0,122917
2,grocery_pos,0,121895
3,shopping_pos,0,115829
4,kids_pets,0,112796
5,shopping_net,0,95830
6,entertainment,0,93781
7,food_dining,0,91310
8,personal_care,0,90538
9,health_fitness,0,85746


In [31]:
df['dob'].min(), df['dob'].max()

(Timestamp('1924-10-30 00:00:00'), Timestamp('2005-01-29 00:00:00'))

In [32]:
df.shape

(1296675, 21)

## Construccion TAD

# Ingenieria de Variables (Adaptacion para Deteccion de Adulto Mayor)

 Adaptaremos la ingenieria de variables para nuestro nuevo objetivo: predecir si el cliente tiene 50 años o mas.
* Se generaran variables de comportamiento, tiempo y distancia.
* Se añadiran proporciones especificas de monto vs hora y monto vs distancia.


In [33]:
# 1. Calcular la Edad temporalmente (años completos)
df['age'] = ((df['trans_date_trans_time'] - df['dob']) / np.timedelta64(1, 'D') / 365.25).astype(int)

# 2.NUEVA VARIABLE OBJETIVO 'y' (1 si es Adulto Mayor >= 50, 0 en caso contrario)
df['y'] = np.where(df['age'] >= 50, 1, 0)

# 3. CUMPLIMIENTO DE REGLA: No colocar la columna edad ni la columna is_fraud dentro del analisis
# Ademas tiramos la fecha de nacimiento (dob) porque con ella el modelo podría inferir la edad fácilmente.
df.drop(columns=['age', 'is_fraud', 'dob'], inplace=True)

In [34]:
# Variables temporales
df['hour'] = df['trans_date_trans_time'].dt.hour
df['dayofweek'] = df['trans_date_trans_time'].dt.dayofweek
df['month'] = df['trans_date_trans_time'].dt.month

# Categoría de montos
def cat_montos(m):
    if m < 15: return 'Muy bajo'
    elif m < 30: return 'Bajo'
    elif m < 100: return 'Medio'
    elif m < 500: return 'Alto'
    else: return 'Muy alto'
df['amt_cat'] = df['amt'].apply(cat_montos)

# Diferencia de tiempo entre la última transacción y la realizada (en minutos)
df = df.sort_values(['cc_num', 'trans_date_trans_time'])
df['trans_date_prev'] = df.groupby('cc_num')['trans_date_trans_time'].shift(1)
df['delta_mins_prev'] = (df['trans_date_trans_time'] - df['trans_date_prev']) / np.timedelta64(1, 'm')

# Distancia euclidiana (ubicación de la transacción vs ubicación del cliente)
df['dist_txn_to_cli'] = np.sqrt((df['lat'] - df['merch_lat'])**2 + (df['long'] - df['merch_long'])**2)

# Distancia entre la última transacción y la actual
df['merch_lat_prev'] = df.groupby('cc_num')['merch_lat'].shift(1)
df['merch_long_prev'] = df.groupby('cc_num')['merch_long'].shift(1)
df['dist_prev_txn'] = np.sqrt(
    (df['merch_lat'] - df['merch_lat_prev'])**2 +
    (df['merch_long'] - df['merch_long_prev'])**2
)

# Diferencias monetarias
df['amt_prev'] = df.groupby('cc_num')['amt'].shift(1)
df['diff_amt_prev'] = df['amt'] - df['amt_prev']

# Longitud del número de tarjeta
df['cc_num_str'] = df['cc_num'].astype(str)
df['cc_num_len'] = df['cc_num_str'].str.len()

# Promedio de monto de transacciones e historial
df['amt_mean_so_far'] = df.groupby('cc_num')['amt'].expanding().mean().reset_index(level=0, drop=True)

# Flag de "nueva categoría"
def flag_nueva_categoria(x):
    seen = set()
    res = []
    for val in x:
        if val not in seen:
            res.append(1)
            seen.add(val)
        else:
            res.append(0)
    return res

df['is_new_category'] = df.groupby('cc_num')['category'].transform(flag_nueva_categoria)

# Montos de gasto acumulado por empleo
df['cum_amt_by_job'] = df.groupby(['job']).apply(lambda group: group['amt'].expanding().sum()).reset_index(level=0, drop=True)

# Construcción de Nuevas Variables Proporcionales

In [36]:
# 1. Proporción entre la cantidad de la transacción y la hora del día.
# Colocar 0 en caso de ser la primera hora del día (hour == 0).
df['prop_amt_hour'] = np.where(df['hour'] == 0, 0, df['amt'] / df['hour'])

# 2. Proporción entre la cantidad de la transacción y la distancia euclidiana (dist_txn_to_cli).
# Añadimos protección por si la distancia es 0 para evitar divisiones infinitas.
df['prop_amt_dist'] = np.where(df['dist_txn_to_cli'] == 0, 0, df['amt'] / df['dist_txn_to_cli'])

# Validación de consistencia
assert df.shape[0] == len(df), "Se agregaron o eliminaron registros del dataframe original."

# Selección de Matriz Final

In [37]:
df.head(2)

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,trans_num,merch_lat,merch_long,y,hour,dayofweek,month,amt_cat,trans_date_prev,delta_mins_prev,dist_txn_to_cli,merch_lat_prev,merch_long_prev,dist_prev_txn,amt_prev,diff_amt_prev,cc_num_str,cc_num_len,amt_mean_so_far,is_new_category,cum_amt_by_job,prop_amt_hour,prop_amt_dist
1017,2019-01-01 12:47:15,60416207185,"fraud_Jones, Sawayn and Romaguera",misc_net,7.2700,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,WY,82514,43.0048,-108.8964,1645,Information systems manager,98e3dcf98101146a577f85a34e58feec,43.9747,-109.7419,0,12,1,1,Muy bajo,NaT,NaN,1.2867,NaN,NaN,NaN,NaN,NaN,60416207185,11,7.2700,1,7.2700,0.6058,5.6501
2724,2019-01-02 08:44:57,60416207185,fraud_Berge LLC,gas_transport,52.9400,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,WY,82514,43.0048,-108.8964,1645,Information systems manager,498120fc45d277f7c88e3dba79c33865,42.0188,-109.0442,0,8,2,1,Medio,2019-01-01 12:47:15,1197.7000,0.9970,43.9747,-109.7419,2.0767,7.2700,45.6700,60416207185,11,30.1050,1,60.2100,6.6175,53.0969


In [38]:
df.shape

(1296675, 39)

In [39]:
df.dtypes

trans_date_trans_time    datetime64[us]
cc_num                            int64
merchant                            str
category                            str
amt                             float64
first                               str
last                                str
gender                              str
street                              str
city                                str
state                               str
zip                                 str
lat                             float64
long                            float64
city_pop                          int64
job                                 str
trans_num                           str
merch_lat                       float64
merch_long                      float64
y                                 int64
hour                              int32
dayofweek                         int32
month                             int32
amt_cat                             str
trans_date_prev          datetime64[us]


Funcion base

In [43]:

from sklearn.preprocessing import KBinsDiscretizer
from sklearn.impute import SimpleImputer


#. Proporción: cantidad de transacción (amt) vs hora del día (hour)
df['hour_num'] = pd.to_numeric(df['hour'], errors='coerce')
df['prop_amt_hour'] = np.where(df['hour_num'] == 0, 0, df['amt'] / df['hour_num'])
df.drop(columns=['hour_num'], inplace=True)

#. Proporción: cantidad de transacción (amt) vs distancia euclidiana (dist_txn_to_cli)
df['prop_amt_dist'] = np.where(df['dist_txn_to_cli'] == 0, 0, df['amt'] / df['dist_txn_to_cli'])

varc = ['merchant','category','first','last','gender','street','city','state','zip','job','hour','dayofweek','month', 
        'amt_cat', 'cc_num_str','is_new_category','cat_amt','month_desc','dayofweek_desc'] 

vard = ['amt','lat','long','city_pop','merch_lat','merch_long', 'delta_mins_prev',
       'dist_txn_to_cli','merch_lat_prev','merch_long_prev','dist_prev_txn','amt_prev','diff_amt_prev','diff_time_prev',
       'cc_num_len','amt_mean_so_far','cum_amt_by_job','dist_lat_long','time_trans','dist_trans',
       'ratio_dist_trans_time_trans','ratio_time_trans_dist_trans','ratio_amt_dist','job_min','job_mean','job_max',
       'prop_amt_hour', 'prop_amt_dist']

# Aplicamos los prefijos
df.rename(columns={c:'v_'+c for c in vard if c in df.columns}, inplace=True)
df.rename(columns={c:'c_'+c for c in varc if c in df.columns}, inplace=True)

varc = [c for c in df.columns if c.startswith('c_')]
vard = [c for c in df.columns if c.startswith('v_')]

print("Variables creadas y prefijos c_ y v_ aplicados correctamente.")

Variables creadas y prefijos c_ y v_ aplicados correctamente.


Normalizamos y discretizamos

In [44]:
def normalizar(df, v, umbral):
    aux = df[v].value_counts(True).to_frame()
    aux[f"n_{v}"] = np.where(aux['proportion'] < umbral, 'CAT_PEQUE', aux.index)
    moda = aux.head()[f'n_{v}'].values[0]
    if aux.loc[aux[f'n_{v}'] == 'CAT_PEQUE']['proportion'].sum() < umbral:
        aux[f'n_{v}'].replace({'CAT_PEQUE': moda}, inplace=True)
    aux.reset_index(inplace=True)
    return df.merge(aux, left_on=[v], right_on=[v], how='inner').drop('proportion', axis=1)

def discretizar(df_aux, v, k):
    kb = KBinsDiscretizer(n_bins=k, encode='ordinal', strategy='quantile', subsample=None)
    df_aux = df_aux.copy()
    
    df_aux[v] = pd.to_numeric(df_aux[v], errors='coerce')
    df_aux = df_aux.dropna(subset=[v])
    
    kb.fit(df_aux[[v]])
    bins = kb.bin_edges_[0] 
    df_aux[f'd_{v}_{k}'] = pd.cut(df_aux[v], bins=bins, include_lowest=True).astype(str)
    return df_aux

In [45]:
umbral = 0.03 
for v in varc:
    df = normalizar(df, v, umbral)

varn = [c for c in df.columns if c.startswith('n_')]

# Filtrar las unarias (variables con 1 solo valor no le sirven al modelo)
unarias = [v for v, conteo in zip(varn, df[varn].nunique()) if conteo <= 1]
varn = [c for c in varn if c not in unarias]
print(f"Variables categóricas normalizadas útiles: {len(varn)}")

/tmp/ipykernel_3454/3821431556.py:6: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  aux[f'n_{v}'].replace({'CAT_PEQUE': moda}, inplace=True)
/tmp/ipykernel_3454/3821431556.py:6: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never 

Variables categóricas normalizadas útiles: 8


Imputamos nulos

In [46]:
df[vard] = df[vard].replace([np.inf, -np.inf], np.nan)

# Imputamos usando la mediana para no romper la discretización
imputador = SimpleImputer(strategy='median')
df[vard] = imputador.fit_transform(df[vard])
print("Imputación de numéricas finalizada.")

Imputación de numéricas finalizada.


discretizamos

In [47]:
%%time
for v in vard:
    for k in range(3, 7):
        if df[v].nunique() < k:
            print(f"Saltando {v} con {k} bins porque tiene pocos valores únicos")
            continue
        
        df = discretizar(df, v, k)

/home/alex/DiplomadoCD/Modulo1/diplomado_env/lib/python3.12/site-packages/sklearn/preprocessing/_discretization.py:385: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 0 are removed. Consider decreasing the number of bins.
  warnings.warn(
/home/alex/DiplomadoCD/Modulo1/diplomado_env/lib/python3.12/site-packages/sklearn/preprocessing/_discretization.py:385: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 0 are removed. Consider decreasing the number of bins.
  warnings.warn(
/home/alex/DiplomadoCD/Modulo1/diplomado_env/lib/python3.12/site-packages/sklearn/preprocessing/_discretization.py:385: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 0 are removed. Consider decreasing the number of bins.
  warnings.warn(


CPU times: user 3min 13s, sys: 40.5 s, total: 3min 54s
Wall time: 3min 55s


vamo a evaluar nuestras variables

In [53]:
from scipy.stats import ks_2samp
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split

if 'index' not in df.columns:
    df = df.reset_index()



In [54]:
# 1. Definimos las unidades muestrales y el target actual
um = ['index']
tgt = ['y']



In [55]:
# 2. Funciones de IV y WOE (con protección +0.0001 para evitar logaritmos indefinidos)
def calculo_iv(df_aux, v, tgt, um):
    aux = df_aux.pivot_table(index=v, columns=tgt, values=um[0], aggfunc='count', fill_value=0)
    aux[list(range(2))] = aux / aux.apply(np.sum)
    aux['w'] = np.log((aux[0] + 0.0001) / (aux[1] + 0.0001))      
    aux['iv'] = (aux[0] - aux[1]) * aux['w']
    return v, aux['iv'].sum()

def clasificacion_woe(df_aux, v, tgt, um):
    aux = df_aux.pivot_table(index=v, columns=tgt, values=um[0], aggfunc='count', fill_value=0)
    aux[list(range(2))] = aux / aux.apply(np.sum)
    aux['w'] = np.log((aux[0] + 0.0001) / (aux[1] + 0.0001))
    aux.drop(range(2), axis=1, inplace=True)
    return v, aux.to_dict()['w']

# 3. Prueba KS (Comparación de variable original vs imputada)
vard_actuales = [v for v in vard if v in df.columns]
ks = pd.DataFrame(map(lambda v: (v, ks_2samp(df[v].dropna(), df[v]).statistic), vard_actuales), columns=['Variable','KS'])
print("--- Kolmogorov-Smirnov ---")
display(ks.head())

# 4. Varianza (Filtro para evitar constantes)
vt = VarianceThreshold(threshold=.1)
vt.fit(df[vard_actuales])
print(f"Variables continuas que superan el umbral de varianza: {sum(vt.get_support())} de {len(vard_actuales)}")

--- Kolmogorov-Smirnov ---


,Variable,KS
0,v_amt,0.0000
1,v_lat,0.0000
2,v_long,0.0000
3,v_city_pop,0.0000
4,v_merch_lat,0.0000


Variables continuas que superan el umbral de varianza: 17 de 18


In [56]:
# Identificamos todas las variables discretizadas (empiezan con d_)
vari = [col for col in df.columns if col.startswith('d_')]

# Calculamos el IV para las variables discretizadas contra nuestro nuevo target 'y'
iv = pd.DataFrame(map(lambda v: calculo_iv(df, v, tgt, um), vari), columns=['variable','IV']).sort_values('IV', ascending=False)

print("--- Top 10 Variables por Information Value (IV) ---")
display(iv.head(10))

# Seleccionamos las mejores variables (usualmente IV > 0.02)
best = iv.loc[iv['IV'] > 0.02, 'variable'].tolist()
if len(best) == 0: 
    print("Ninguna superó 0.02. Tomando el top 10...")
    best = iv['variable'].head(10).tolist()
    
# Verificamos el poder predictivo de la mejor variable discretizada encontrada
mejor_variable = best[0]
print(f"\nEjemplo de Poder Predictivo con la mejor variable: {mejor_variable}")
display(df[[tgt[0], mejor_variable]].value_counts().reset_index())

--- Top 10 Variables por Information Value (IV) ---


,variable,IV
57,d_v_amt_mean_so_far_4,1.1705
59,d_v_amt_mean_so_far_6,1.1077
58,d_v_amt_mean_so_far_5,1.1036
56,d_v_amt_mean_so_far_3,0.8101
13,d_v_city_pop_4,0.1405
15,d_v_city_pop_6,0.1398
14,d_v_city_pop_5,0.1311
12,d_v_city_pop_3,0.1231
27,d_v_delta_mins_prev_6,0.0853
26,d_v_delta_mins_prev_5,0.0832



Ejemplo de Poder Predictivo con la mejor variable: d_v_amt_mean_so_far_4


,y,d_v_amt_mean_so_far_4,count
0,0,"(83.851, 1433.54]",304688
1,0,"(1.0290000000000001, 58.023]",251508
2,1,"(58.023, 65.047]",183636
3,1,"(65.047, 83.851]",181192
4,0,"(65.047, 83.851]",142977
5,0,"(58.023, 65.047]",140533
6,1,"(1.0290000000000001, 58.023]",72661
7,1,"(83.851, 1433.54]",19480


Particion

In [57]:
# Filtramos nuestro dataset con la unidad muestral, el target y las mejores variables
X = df[um + best]
y = df[um + tgt]

# Partición 70/30
Xt, Xv, yt, yv = train_test_split(X, y, train_size=.7, random_state=42)

varw = [] # Aquí guardaremos los nuevos nombres con prefijo x_

# Mapeo WOE
for v in best:
    v_name, woe_dict = clasificacion_woe(df, v, tgt, um)
    
    # Creamos el nombre final con el prefijo "x_"
    nombre_final_x = f"x_{v}"
    varw.append(nombre_final_x)
    
    # Aplicamos el diccionario al train y validation
    Xt[nombre_final_x] = Xt[v].map(woe_dict)
    Xv[nombre_final_x] = Xv[v].map(woe_dict)
    
    # Llenar nulos por si acaso un valor en Test no existía en Train
    Xt[nombre_final_x] = Xt[nombre_final_x].fillna(0)
    Xv[nombre_final_x] = Xv[nombre_final_x].fillna(0)

print(f"Variables transformadas a WOE y renombradas a x_: {len(varw)}")

Variables transformadas a WOE y renombradas a x_: 19


TAD

In [58]:
# Unimos las características transformadas WOE (varw) con sus etiquetas correspondientes
tad_t = Xt[um + varw].merge(yt, on=um, how='inner')
tad_v = Xv[um + varw].merge(yv, on=um, how='inner')

# Mostramos el encabezado de nuestra Tabla Analítica de Datos de entrenamiento
display(tad_t.head())



,index,x_d_v_amt_mean_so_far_4,x_d_v_amt_mean_so_far_6,x_d_v_amt_mean_so_far_5,x_d_v_amt_mean_so_far_3,x_d_v_city_pop_4,x_d_v_city_pop_6,x_d_v_city_pop_5,x_d_v_city_pop_3,x_d_v_delta_mins_prev_6,x_d_v_delta_mins_prev_5,x_d_v_cum_amt_by_job_6,x_d_v_cum_amt_by_job_5,x_d_v_delta_mins_prev_4,x_d_v_cum_amt_by_job_4,x_d_v_delta_mins_prev_3,x_d_v_cum_amt_by_job_3,x_d_v_prop_amt_hour_5,x_d_v_prop_amt_hour_4,x_d_v_prop_amt_hour_6,y
0,992821,0.6329,-0.2466,-0.5291,0.2870,0.1330,0.4222,0.3336,0.4792,0.4355,0.4225,0.2287,0.0151,0.3988,0.2344,0.3537,0.0321,-0.3133,-0.2488,-0.1989,0
1,833489,-0.8450,0.5652,-0.1487,1.1594,0.1330,0.0897,0.3336,-0.0323,-0.2101,-0.1871,-0.3815,-0.3707,-0.3660,-0.3362,-0.3262,-0.3189,-0.3133,0.1622,-0.1989,0
2,517107,-0.8756,-1.0384,-1.2412,-1.1041,0.5732,0.4222,0.3336,0.4792,0.4355,0.4225,0.3100,0.3147,0.3988,0.3404,0.3537,0.3163,0.0802,0.1870,0.0046,1
3,166051,-0.8450,0.5652,-0.1487,1.1594,-0.1510,-0.1494,-0.2293,-0.0323,0.4355,0.4225,0.3225,0.3364,0.3988,0.3404,0.3537,0.3163,0.1416,0.1870,0.2621,0
4,473161,-0.8450,-1.1702,-0.1487,-1.1041,-0.4615,-0.5601,-0.5141,-0.3880,0.2746,0.2185,0.3100,0.3147,0.1356,0.3404,0.3537,0.3163,0.0802,0.1870,0.2621,0


Exportar TAD

In [59]:
nombre_archivo = 'GARIBAY_MORALES_ALEJANDRO_T4.parquet'
tad_t.to_parquet(nombre_archivo, index=False)

print(f"TAD dimensiones: {tad_t.shape}")

TAD dimensiones: (907672, 21)
